In [1]:
!nvidia-smi

Tue Mar  3 23:57:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install vLLM + utilities (vLLM pulls in torch/transformers)
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 

In [3]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

print("Loading Financial PhraseBank...")

# FinanceMTEB version has a clean held-out test split (1,000 examples)
ds_full = load_dataset("FinanceMTEB/financial_phrasebank")
ds = ds_full["test"]

# Labels: 0 = negative, 1 = neutral, 2 = positive
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
INT_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

print(f"Dataset size: {len(ds)} examples")
print(f"Label distribution: {dict(zip(*np.unique(ds['label'], return_counts=True)))}")

Loading Financial PhraseBank...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/104k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/80.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1264 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset size: 1000 examples
Label distribution: {np.int64(0): np.int64(138), np.int64(1): np.int64(612), np.int64(2): np.int64(250)}


In [4]:
# --- Config ---
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS = 64
EVAL_SIZE = len(ds)     # set to e.g. 50 for a quick smoke test
ENABLE_THINKING = False  # recommended False for fast classification

SYSTEM_PROMPT = (
    "You are a financial sentiment classifier. "
    "Classify the sentiment of the following financial news sentence as exactly one of: "
    "positive, negative, or neutral. "
    "Respond with only the single word label."
)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "enable_thinking": ENABLE_THINKING,
})

{'model': 'Qwen/Qwen3-8B', 'eval_size': 1000, 'max_new_tokens': 64, 'enable_thinking': False}


In [6]:
if __name__ == '__main__':
  from transformers import AutoTokenizer
  from vllm import LLM, SamplingParams

  print("Loading tokenizer...")
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

  print("Loading Qwen3-8B with vLLM...")
  print("  PagedAttention: enabled (automatic)")
  print("  Cont. batching: enabled (automatic)")

  llm = LLM(
      model=MODEL_NAME,
      dtype="float16",
      gpu_memory_utilization=0.95,
      max_model_len=4096,
      enforce_eager=False,
  )

  sampling_params = SamplingParams(
      temperature=0,
      top_k=20,
      max_tokens=MAX_NEW_TOKENS,
  )

  print("Model loaded successfully!")

Loading tokenizer...
Loading Qwen3-8B with vLLM...
  PagedAttention: enabled (automatic)
  Cont. batching: enabled (automatic)
INFO 03-04 00:03:28 [utils.py:223] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}
INFO 03-04 00:03:28 [model.py:529] Resolved architecture: Qwen3ForCausalLM
WARNING 03-04 00:03:28 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-04 00:03:28 [model.py:1549] Using max model len 4096
INFO 03-04 00:03:28 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-04 00:05:27 [llm.py:355] Supported tasks: ['generate']
Model loaded successfully!


In [7]:
# --- Output directory (Colab Drive if available) ---
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_SAVE_DIR = '/content/drive/MyDrive/Colab_Data/Qwen3_8B_FinancialPhraseBank_Eval_vLLM'
else:
    DRIVE_SAVE_DIR = os.path.abspath('./qwen3_8b_finpb_eval_vllm_outputs')

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Colab_Data/Qwen3_8B_FinancialPhraseBank_Eval_vLLM


In [8]:
# --- Checkpoint setup ---
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, 'qwen3_8b_finpb_vllm_checkpoint.jsonl')
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, 'qwen3_8b_finpb_vllm_raw_outputs.json')

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

sentences   = ds["text"]
int_labels  = ds["label"]
str_labels  = [LABEL_MAP[l] for l in int_labels]

already_done = len(results)
remaining_sentences = sentences[already_done:EVAL_SIZE]
remaining_gt        = str_labels[already_done:EVAL_SIZE]
print(f"{already_done} already done, {len(remaining_sentences)} remaining.")

def build_prompt(sentence: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sentence},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

prompts = [build_prompt(s) for s in remaining_sentences]
print(f"Built {len(prompts)} prompts (enable_thinking={ENABLE_THINKING}).")
print("\nExample prompt (first 400 chars):")
print(prompts[0][:400] if prompts else "(none — all done)")

No checkpoint found — starting from scratch.
0 already done, 1000 remaining.
Built 1000 prompts (enable_thinking=False).

Example prompt (first 400 chars):
<|im_start|>system
You are a financial sentiment classifier. Classify the sentiment of the following financial news sentence as exactly one of: positive, negative, or neutral. Respond with only the single word label.<|im_end|>
<|im_start|>user
Rautaruukki said construction group YIT has awarded it a 2.5 mln eur contract to supply the steel structures for a new bridge spanning the Kemijoki river in


In [9]:
# --- Generate with vLLM ---
# vLLM does continuous batching internally; we can pass all prompts at once.

if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")
    print("(vLLM will log per-step throughput below)")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec" if throughput is not None else "  Throughput: N/A")

    raw_cache = [
        {
            "idx": already_done + i,
            "sentence": remaining_sentences[i],
            "ground_truth": remaining_gt[i],
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs (if present).")
    if not os.path.exists(OUTPUTS_CACHE):
        raw_cache = []
    else:
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

Generating responses for 1000 prompts...
(vLLM will log per-step throughput below)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


Generation complete.
  Time:       0.3 min
  Tokens:     2,000
  Throughput: 126.8 tokens/sec
Raw outputs cached to: /content/drive/MyDrive/Colab_Data/Qwen3_8B_FinancialPhraseBank_Eval_vLLM/qwen3_8b_finpb_vllm_raw_outputs.json
Raw items available for scoring: 1000


In [10]:
from sklearn.metrics import classification_report, confusion_matrix

def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text

def extract_label(text: str):
    text = text.strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in text:
            return label
    return None

# --- Score new outputs and append to checkpoint ---
new_results = []
for item in raw_cache:
    idx = item["idx"]
    response = item["response"]

    answer_text = strip_thinking(response) if ENABLE_THINKING else response
    pred = extract_label(answer_text)
    gt = item["ground_truth"]

    res = {
        "idx": idx,
        "sentence": item.get("sentence"),
        "ground_truth": gt,
        "prediction": pred,
        "raw_response": response.strip(),
        "is_correct": (pred == gt),
        "is_unknown": (pred is None),
        "new_tokens": int(item.get("n_tokens", 0)),
    }
    new_results.append(res)

# Write new results (and extend in-memory results)
if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

# --- Final metrics ---
all_gt = [r["ground_truth"] for r in results]
all_pred = [r["prediction"] if r["prediction"] is not None else "unknown" for r in results]

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)

final_metrics = {
    "method": f"0_shot_vllm_thinking={ENABLE_THINKING}",
    "model": MODEL_NAME,
    "dataset": "FinanceMTEB/financial_phrasebank:test",
    "eval_size": len(results),
    "accuracy": correct_count / len(results) if results else 0,
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": (gen_time / 60) if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

print("\n=== Per-Class Report ===")
labels = ["negative", "neutral", "positive"]
print(classification_report(all_gt, all_pred, labels=labels))

print("=== Confusion Matrix (rows=true, cols=pred) ===")
print(labels)
print(confusion_matrix(all_gt, all_pred, labels=labels))

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")

Total scored results: 1000

=== Final Metrics ===
  method: 0_shot_vllm_thinking=False
  model: Qwen/Qwen3-8B
  dataset: FinanceMTEB/financial_phrasebank:test
  eval_size: 1000
  accuracy: 0.963
  unknown_frac: 0.0
  total_new_tokens: 2000
  gen_tokens_per_sec: 126.75943410638794
  total_gen_time_min: 0.2629653056462606

=== Per-Class Report ===
              precision    recall  f1-score   support

    negative       0.93      1.00      0.97       138
     neutral       0.98      0.97      0.97       612
    positive       0.94      0.93      0.94       250

    accuracy                           0.96      1000
   macro avg       0.95      0.97      0.96      1000
weighted avg       0.96      0.96      0.96      1000

=== Confusion Matrix (rows=true, cols=pred) ===
['negative', 'neutral', 'positive']
[[138   0   0]
 [  5 592  15]
 [  5  12 233]]

Metrics saved to: /content/drive/MyDrive/Colab_Data/Qwen3_8B_FinancialPhraseBank_Eval_vLLM/final_metrics.json
